# Loading and Exploring ODMR Data with qdmpy

This notebook demonstrates how to load and explore ODMR (Optically Detected Magnetic Resonance) data from a QDM (Quantum Diamond Microscope) measurement using qdmpy.

We use the `MIL2_FOV1` test dataset, which contains:
- Two MATLAB files (`run_00000.mat`, `run_00001.mat`) with ODMR spectra at two field polarities
- LED and laser reference images
- Acquisition metadata in header files

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qdmpy.fitting import FitManager
from qdmpy.odmr import ODMR, ODMRData, b111_from_dip_positions
from qdmpy.odmr.io import MatlabLoader
from qdmpy.odmr.processors import (
    BinningProcessor,
    FluorescenceCorrectionProcessor,
    NormalizationProcessor,
)

## 1. Load raw ODMR data

The `MatlabLoader` reads `run_*.mat` files from the data folder. Each file corresponds to
one field polarity and contains two image stacks (low-field and high-field frequency ranges)
with 51 frequency steps across a 1920 × 1200 pixel field of view.

In [ ]:
DATA_FOLDER = Path.home() / "git" / "qdmpy-core" / "tests" / "data" / "MIL2_FOV1"

loader = MatlabLoader(data_folder=str(DATA_FOLDER))
odmr_data = ODMRData.from_loader(loader)

## 2. Inspect the xarray DataArray

The data is stored as a 5D `xarray.DataArray` with named dimensions:
- `polarity` — field polarity (one per `.mat` file)
- `freq_range` — low-field vs high-field frequency band
- `y`, `x` — spatial pixel coordinates
- `freq_idx` — frequency sweep index

Actual frequency values in GHz are stored as a non-dimension coordinate `freq_ghz`.

In [ ]:
odmr_data.data

In [ ]:
# Frequency values for each range (in GHz)
freq_ghz = odmr_data.data.coords["freq_ghz"].values  # shape (n_frange, n_freq)
frange_labels = list(odmr_data.data.coords["freq_range"].values)

for _label, freqs in zip(frange_labels, freq_ghz, strict=False):
    step_mhz = (freqs[-1] - freqs[0]) / (len(freqs) - 1) * 1000

## 3. Load reference images

The dataset includes LED (white light) and laser images used for alignment and fluorescence correction.

In [ ]:
led_image = np.loadtxt(DATA_FOLDER / "LED.csv")
laser_image = np.loadtxt(DATA_FOLDER / "laser.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(led_image, cmap="gray")
axes[0].set_title("LED (white light) image")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(laser_image, cmap="bone")
axes[1].set_title("Laser image")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 4. Visualize mean ODMR spectra

Before any processing, let's look at the spatially-averaged ODMR spectra for each
polarity and frequency range. Each spectrum should show Lorentzian dips at the
NV center resonance frequencies.

In [ ]:
mean_spectra = odmr_data.data.mean(dim=["y", "x"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
pol_labels = list(odmr_data.data.coords["polarity"].values)

for i_frange in range(odmr_data.data.sizes["freq_range"]):
    ax = axes[i_frange]
    freqs = freq_ghz[i_frange]
    for i_pol in range(odmr_data.data.sizes["polarity"]):
        spectrum = mean_spectra.isel(polarity=i_pol, freq_range=i_frange).values
        ax.plot(freqs, spectrum, "o-", markersize=3, label=pol_labels[i_pol])
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Fluorescence (counts)")
    ax.set_title(f"Frequency range {i_frange} (mean over all pixels)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Apply processing pipeline

qdmpy uses a processor pipeline to prepare data for fitting:
1. **Spatial binning** — reduces noise by averaging adjacent pixels (e.g. 4×4)
2. **Normalization** — divides each spectrum by its maximum value so dip depths represent contrast
3. **Fluorescence correction** — subtracts spatially-varying background fluorescence

In [ ]:
odmr = ODMR(odmr_data)

odmr.processor_manager.add_processor(BinningProcessor(bin_factor=4))
odmr.processor_manager.add_processor(NormalizationProcessor(method="mean"))
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor())
odmr.process_data()

## 6. Compare raw vs processed spectra

After binning and normalization, the spectra are smoother and the dips are easier
to identify. The y-axis now represents normalized contrast (1 = baseline).

In [ ]:
proc = odmr.processed_data

# Pick a pixel near the center of the binned image
cy, cx = proc.data.sizes["y"] // 2, proc.data.sizes["x"] // 2

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i_pol in range(proc.data.sizes["polarity"]):
    for i_frange in range(proc.data.sizes["freq_range"]):
        ax = axes[i_pol, i_frange]
        freqs = freq_ghz[i_frange]

        spectrum = proc.data.isel(polarity=i_pol, freq_range=i_frange, y=cy, x=cx).values
        ax.plot(freqs, spectrum, "o-", markersize=3, color="C0")

        ax.set_xlabel("Frequency (GHz)")
        ax.set_ylabel("Normalized intensity")
        ax.set_title(f"pol={i_pol}, frange={i_frange} | pixel ({cy}, {cx})")
        ax.grid(True, alpha=0.3)

plt.suptitle("Processed ODMR spectra (4×4 binned, normalized)", y=1.01)
plt.tight_layout()
plt.show()

## 7. Spatial maps of fluorescence and contrast

We can create spatial maps by reducing the frequency dimension. The mean gives an overall
fluorescence map, while the contrast (max − min along frequency) highlights regions with
strong ODMR signal.

In [ ]:
# Use raw (unbinned) data for full-resolution images
raw = odmr_data.data

# Fluorescence map: mean over frequency, averaged over polarities and ranges
fluorescence = raw.mean(dim=["polarity", "freq_range", "freq_idx"])

# Contrast map: (max - min) / max along frequency, averaged over pol/frange
spec_max = raw.max(dim="freq_idx")
spec_min = raw.min(dim="freq_idx")
contrast = ((spec_max - spec_min) / spec_max).mean(dim=["polarity", "freq_range"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(fluorescence.values, cmap="gray")
axes[0].set_title("Mean fluorescence")
axes[0].set_xlabel("x (pixels)")
axes[0].set_ylabel("y (pixels)")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="Counts")

im1 = axes[1].imshow(contrast.values, cmap="hot")
axes[1].set_title("ODMR contrast (dip depth / max)")
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="Contrast")

plt.tight_layout()
plt.show()

## 8. Interactive pixel exploration

Select a few pixels across the field of view and compare their spectra to see how the
ODMR response varies spatially.

In [ ]:
# Sample pixels at different locations in the binned data
ny, nx = proc.data.sizes["y"], proc.data.sizes["x"]
sample_pixels = [
    (ny // 4, nx // 4),
    (ny // 4, 3 * nx // 4),
    (3 * ny // 4, nx // 4),
    (3 * ny // 4, 3 * nx // 4),
    (ny // 2, nx // 2),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(sample_pixels)))

for i_frange in range(proc.data.sizes["freq_range"]):
    ax = axes[i_frange]
    freqs = freq_ghz[i_frange]
    for (py, px), color in zip(sample_pixels, colors, strict=False):
        spectrum = proc.data.isel(polarity=0, freq_range=i_frange, y=py, x=px).values
        ax.plot(freqs, spectrum, "-", color=color, alpha=0.8, label=f"({py}, {px})")
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Normalized intensity")
    ax.set_title(f"Frequency range {i_frange}")
    ax.legend(fontsize=8, title="(y, x)")
    ax.grid(True, alpha=0.3)

plt.suptitle("ODMR spectra at selected pixels (pol_0)", y=1.01)
plt.tight_layout()
plt.show()

## 9. Dip position map (quick estimate)

As a simple estimate of the resonance frequency at each pixel, we find the frequency index
of the minimum value. This gives a rough spatial map of resonance shifts without full fitting.

In [ ]:
# Work with processed (binned + normalized) data
proc_da = proc.data
pols = list(proc_da.coords["polarity"].values)       # ['neg', 'pos']
franges = list(proc_da.coords["freq_range"].values)   # ['low', 'high']
freq_ghz_proc = proc_da.coords["freq_ghz"].values     # shape (n_frange, n_freq)

# Store dip frequencies for each (pol, frange) — reused for quick B₁₁₁ estimate below
dip_freq_maps: dict = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i_pol, pol in enumerate(pols):
    for i_frange, frange in enumerate(franges):
        ax = axes[i_pol, i_frange]
        freqs = freq_ghz_proc[i_frange]

        # Index of minimum along frequency axis → map to frequency in GHz
        min_idx = proc_da.sel(polarity=pol, freq_range=frange).argmin(dim="freq_idx").values
        dip_freq = freqs[min_idx]
        dip_freq_maps[(pol, frange)] = dip_freq

        im = ax.imshow(dip_freq, cmap="RdBu_r")
        ax.set_title(f"Dip frequency | pol={pol}, range={frange}")
        ax.set_xlabel("x (binned pixels)")
        ax.set_ylabel("y (binned pixels)")
        fig.colorbar(im, ax=ax, shrink=0.8, label="GHz")

plt.suptitle("Dip position maps (argmin estimate, no fitting)", y=1.01)
plt.tight_layout()
plt.show()

## 10. Quick B₁₁₁ estimate from dip positions

Without full spectral fitting, a rough magnetic field map is obtained directly from the
argmin dip positions above.

For each pixel the Zeeman splitting between the high- and low-frequency branches gives:

```
δB[pol] = sign[pol] × (f_high − f_low) / 2 / γ_NV    [µT]
```

where `γ_NV = GAMMA_NV = 28.024 GHz/T` and `sign = {neg: −1, pos: +1}`.

Combining the two field polarities separates remanent from induced components:
- **Remanent field**: `(δB_neg + δB_pos) / 2` — permanent magnetization
- **Induced field**: `(δB_neg − δB_pos) / 2` — paramagnetic / bias-tracking component

In [ ]:
b111 = b111_from_dip_positions(proc_da)
b111_remanent_quick = b111["remanent"]
b111_induced_quick  = b111["induced"]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmax = np.percentile(np.abs(b111_remanent_quick), 99)
im0 = axes[0].imshow(b111_remanent_quick, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("B₁₁₁ remanent — quick estimate (µT)")
axes[0].set_xlabel("x (binned pixels)")
axes[0].set_ylabel("y (binned pixels)")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="µT")

vmax = np.percentile(np.abs(b111_induced_quick), 99)
im1 = axes[1].imshow(b111_induced_quick, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("B₁₁₁ induced — quick estimate (µT)")
axes[1].set_xlabel("x (binned pixels)")
axes[1].set_ylabel("y (binned pixels)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="µT")

plt.suptitle("Quick B₁₁₁ from dip positions (µT)", y=1.01)
plt.tight_layout()
plt.show()

## 11. Full spectral fitting with FitManager

The quick estimate gives a useful first impression. For quantitative field maps,
`FitManager` fits each pixel's full ODMR spectrum to a physics-based model (`ESR14N`:
three Lorentzian dips with ¹⁴N hyperfine splitting at 2.16 MHz).

This gives accurate resonance frequencies, linewidths, contrasts, and chi² fit quality per pixel.

In [ ]:
fitm = FitManager("ESR14N")
res = fitm.fit(proc_da, freq_ghz_proc)

metrics = res.get_fit_quality_metrics()

## 12. B₁₁₁ from full fit

`FitResult` computes B₁₁₁ from the fitted resonance centres — more accurate than the argmin
estimate since the Lorentzian fit averages over the full spectrum and is robust to noise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmax = np.percentile(np.abs(res.b111_remanent), 99)
im0 = axes[0].imshow(res.b111_remanent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("B₁₁₁ remanent (µT) — full fit")
axes[0].set_xlabel("x (pixels)")
axes[0].set_ylabel("y (pixels)")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="µT")

vmax = np.percentile(np.abs(res.b111_induced), 99)
im1 = axes[1].imshow(res.b111_induced, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("B₁₁₁ induced (µT) — full fit")
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="µT")

plt.suptitle("B₁₁₁ from ESR14N full fit", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Fit quality: chi² map (lower = better fit)
chi2_map = res.parameters["chi2"][0, 0].reshape(res.scan_dimensions)

fig, ax = plt.subplots(figsize=(7, 5))
vmax = np.percentile(chi2_map, 99)
im = ax.imshow(chi2_map, cmap="hot_r", vmin=0, vmax=vmax)
ax.set_title("χ² map (neg polarity, low range)")
ax.set_xlabel("x (pixels)")
ax.set_ylabel("y (pixels)")
fig.colorbar(im, ax=ax, shrink=0.8, label="χ²")
plt.tight_layout()
plt.show()

## 13. Comparing fit backends: gpufit vs scipy

`FitManager` fits through an injectable `FitBackend` (QEP-068): a CUDA-accelerated
`pygpufit` backend, and a pure-Python `scipy` backend that runs anywhere without a GPU.
Both implement the same interface, so switching is just a constructor argument.

scipy fits each pixel independently in Python, so we compare on a small patch
(20×20 pixels) rather than the full binned image to keep runtime reasonable —
the same comparison holds at full scale, just slower for scipy.

In [ ]:
import time

patch = proc_da.isel(y=slice(0, 20), x=slice(0, 20))

backend_results = {}
for backend_name in ["gpufit", "scipy"]:
    fitm_b = FitManager("ESR14N", backend=backend_name)
    t0 = time.perf_counter()
    res_b = fitm_b.fit(patch, freq_ghz_proc)
    elapsed = time.perf_counter() - t0
    backend_results[backend_name] = {"result": res_b, "time_s": elapsed}
    chi2_mean = res_b.get_fit_quality_metrics()["mean_chi2"]
    print(f"{backend_name:>8}: {elapsed:6.2f} s  (mean_chi2={chi2_mean:.4f})")

In [ ]:
res_gpufit = backend_results["gpufit"]["result"]
res_scipy = backend_results["scipy"]["result"]

center_diff_mhz = np.abs(res_gpufit.parameters["center"] - res_scipy.parameters["center"]) * 1e3
print(
    f"Center agreement (gpufit vs scipy): max={center_diff_mhz.max():.4f} MHz, "
    f"mean={center_diff_mhz.mean():.4f} MHz"
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
vmax = np.percentile(np.abs(res_gpufit.b111_remanent), 99)
axes[0].imshow(res_gpufit.b111_remanent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("B₁₁₁ remanent — gpufit")
axes[1].imshow(res_scipy.b111_remanent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("B₁₁₁ remanent — scipy")
diff_im = axes[2].imshow(res_gpufit.b111_remanent - res_scipy.b111_remanent, cmap="RdBu_r")
axes[2].set_title("Difference (gpufit - scipy)")
fig.colorbar(diff_im, ax=axes[2], shrink=0.8, label="µT")
for ax in axes:
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
gpufit_t = backend_results["gpufit"]["time_s"]
scipy_t = backend_results["scipy"]["time_s"]
fig.suptitle(
    f"Backend comparison on a {patch.sizes['y']}x{patch.sizes['x']}-pixel patch "
    f"(gpufit: {gpufit_t:.2f}s, scipy: {scipy_t:.2f}s)",
    y=1.03,
)
fig.tight_layout()
plt.show()

## 14. Summary

This notebook walked through a complete QDM analysis workflow on the MIL2_FOV1 dataset:

| Step | Tool | Key output |
|------|------|-----------|
| Load | `MatlabLoader` + `ODMRData.from_loader` | 5D xarray DataArray (pol, frange, y, x, freq) |
| Process | `ODMR` + `BinningProcessor`, `NormalizationProcessor`, `FluorescenceCorrectionProcessor` | Binned & normalised spectra |
| Quick B₁₁₁ | argmin per pixel + B₁₁₁ formula | Rough field maps without GPU fitting |
| Full fit | `FitManager("ESR14N").fit(data, freq)` | `FitResult` with all parameters |
| B₁₁₁ | `res.b111_remanent`, `res.b111_induced` | Accurate field maps in µT |
| Backend comparison | `FitManager(..., backend="gpufit"/"scipy")` | Confirms gpufit/scipy agree on real data |

The **quick estimate** is useful for a fast sanity-check before committing to a full GPU fit.
The **full fit** gives quantitative parameters (linewidth, contrast, chi²) and more accurate field maps.

### Next steps
- `processor_tutorial.ipynb` — deep dive into the processing pipeline and fluorescence correction
- `tutorial_fitting.ipynb` — constraint management and advanced fitting options
- `tutorial_models.ipynb` — ESR14N / ESR15N / ESRSINGLE model details